Qwen2.5-3B-Instruct + Cricket-only dataset → fine-tuned cricket model

Qwen2.5-3B-Instruct
          ↓
   Cricket Dataset
          ↓
   Chat Formatting
          ↓
      Tokenization
          ↓
     LoRA / QLoRA
          ↓
      Fine-tuning
          ↓
   Cricket-specialized
       Adapter
          ↓



In [6]:
!pip install -q -U unsloth datasets trl transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 134.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
import torch
import os
import json

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
BF16 supported: False


In [9]:
import unsloth

In [10]:
from unsloth import FastLanguageModel

model_name = "Qwen/Qwen2.5-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None
)

print("Qwen model loaded successfully!")

==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen model loaded successfully!


In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42
)

print("LoRA adapters added!")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA adapters added!


In [12]:
os.makedirs("data", exist_ok=True)

cricket_data = [
    {
        "instruction": "What is a yorker in cricket?",
        "input": "",
        "output": "A yorker is a cricket delivery that pitches very close to the batter's feet, usually near the base of the stumps."
    },
    {
        "instruction": "What is a hat-trick in cricket?",
        "input": "",
        "output": "A hat-trick occurs when a bowler dismisses three batters with three consecutive deliveries."
    },
    {
        "instruction": "What does LBW mean in cricket?",
        "input": "",
        "output": "LBW means Leg Before Wicket. It is a method of dismissal where the batter's body prevents the ball from hitting the wicket, subject to the conditions in the Laws of Cricket."
    },
    {
        "instruction": "What is a maiden over?",
        "input": "",
        "output": "A maiden over is an over in which the bowler concedes no runs from the bat or other scoring opportunities, apart from certain penalty situations."
    },
    {
        "instruction": "What is a powerplay in T20 cricket?",
        "input": "",
        "output": "A powerplay is a period during which fielding restrictions apply, limiting the number of fielders allowed outside the designated inner circle."
    },
    {
        "instruction": "What is a googly?",
        "input": "",
        "output": "A googly is a deceptive delivery bowled by a leg-spinner that turns in the opposite direction from what the batter normally expects."
    },
    {
        "instruction": "What is a bouncer?",
        "input": "",
        "output": "A bouncer is a short-pitched delivery that rises sharply toward the batter after bouncing on the pitch."
    },
    {
        "instruction": "What is a Test match?",
        "input": "",
        "output": "A Test match is the longest format of international cricket and is normally played over a maximum of five days."
    }
]

with open("data/cricket.jsonl", "w") as f:
    for item in cricket_data:
        f.write(json.dumps(item) + "\n")

print("Cricket dataset created!")
print("Number of examples:", len(cricket_data))

Cricket dataset created!
Number of examples: 8


In [13]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="data/cricket.jsonl",
    split="train"
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 8
})


In [14]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset["train"]
valid_dataset = dataset["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(valid_dataset))

Training examples: 6
Validation examples: 2


In [15]:
def formatting_func(example):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in cricket."
        },
        {
            "role": "user",
            "content": example["instruction"]
            + (
                "\n\n" + example["input"]
                if example["input"]
                else ""
            )
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

In [23]:
def tokenize_function(example):
    text = formatting_func(example)

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=1024,
        padding="max_length"
    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

In [24]:
tokenized_train = train_dataset.map(
    tokenize_function,
    remove_columns=train_dataset.column_names
)

tokenized_valid = valid_dataset.map(
    tokenize_function,
    remove_columns=valid_dataset.column_names
)

print("Training dataset:")
print(tokenized_train)

print("\nValidation dataset:")
print(tokenized_valid)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Training dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 6
})

Validation dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})


In [25]:
from transformers import TrainingArguments

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir="cricket_qwen_model",

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    num_train_epochs=3,

    logging_steps=1,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,

    optim="adamw_torch",

    lr_scheduler_type="cosine",

    warmup_ratio=0.05,

    report_to="none",

    save_total_limit=2,

    seed=42
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [26]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [27]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,

    data_collator=data_collator,

    processing_class=tokenizer
)

print("Trainer created successfully!")

Trainer created successfully!


In [28]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,4.166248,4.488106
2,4.166248,4.488106
3,4.166248,2.976087


Unsloth: Restored added_tokens_decoder metadata in cricket_qwen_model/checkpoint-1/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in cricket_qwen_model/checkpoint-2/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in cricket_qwen_model/checkpoint-3/tokenizer_config.json.


TrainOutput(global_step=3, training_loss=4.166248321533203, metrics={'train_runtime': 49.054, 'train_samples_per_second': 0.367, 'train_steps_per_second': 0.061, 'total_flos': 310178192818176.0, 'train_loss': 4.166248321533203, 'epoch': 3.0})

In [29]:
output_dir = "cricket_qwen_lora"

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("Cricket fine-tuned model saved to:", output_dir)

Unsloth: Restored added_tokens_decoder metadata in cricket_qwen_lora/tokenizer_config.json.


Cricket fine-tuned model saved to: cricket_qwen_lora


In [30]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "You are a helpful AI assistant specialized in cricket."
    },
    {
        "role": "user",
        "content": "Explain what a yorker is in cricket."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=150,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0][inputs.shape[-1]:],
    skip_special_tokens=True
)

print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In cricket, a yorker is a type of delivery that is bowled very close to the wicket, usually within the first few inches of the pitch. The ball is delivered with such pace and accuracy that it reaches the batsman just as or before they have time to fully swing their bat.

The key characteristics of a yorker are:

1. **Close to the Wicket**: It's thrown at the batsman while they are near to the wicket, often just a few feet away.
2. **Pace and Accuracy**: The bowler uses high speed and precision to get the ball close to the batsman, making it difficult for them to play a shot.
3. **Risk for Batsmen**: A well-bow
